In [20]:
import sys
import os

# Get the absolute path to the parent directory (CSCI5527-FINAL)
parent_dir = os.path.abspath("..")

# Add the parent directory to the system path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [21]:
import torch
import torch.optim as optim
import torch.nn as nn
import segmentation_models_pytorch as smp
from fastai.losses import DiceLoss, CrossEntropyLossFlat

import io
import pandas as pd
from contextlib import redirect_stdout, redirect_stderr
import tqdm

from matrices import *
from preprocessing import *
from utils import *

In [22]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.backends.cudnn.is_available())
n_gpu = torch.cuda.device_count()
print(f"Total GPUs available: {n_gpu}")
for i in range(n_gpu):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using {device} device")
generator = torch.Generator(device=device)

2.11.0+cu126
True
True
Total GPUs available: 1
Device 0: NVIDIA GeForce RTX 2060 SUPER
Using cuda:0 device


In [23]:
def baseline_model_pipeline(base_dir, dict_files):
    # Locate project root and TACK data folders dynamically relative to the script location
    dataset_folder = os.path.join(base_dir, "TACK_Tunnel_Data")

    csv_source_dir = os.path.join(dataset_folder, "2_model_input")
    raw_mask_dir = os.path.join(dataset_folder, "3_mask")

    # Initialize pipeline pointing to the TTD dataset structure
    pipeline = TunnelDataPipeline(
        base_dir=dataset_folder,
        original_mask_dir=raw_mask_dir
    )

    # Define standard training splits for the multi-domain tunnel study
    train_files = dict_files["train_files"]
    val_files = dict_files["val_files"]
    test_files = dict_files["test_files"]

    # Step 1: Load metadata
    print("Loading CSV metadata...")
    df_train_val, df_test = pipeline.load_csv_data(
        csv_source_dir=csv_source_dir,
        train_files=train_files,
        val_files=val_files,
        test_files=test_files
    )

    # Step 2: Sanitize and separate training/validation masks
    print("Sanitizing training and validation masks...")
    df_train_val_ready = pipeline.sanitize_masks(df_train_val, class_pixel_value=40)

    # Step 3: Sanitize test masks
    print("Sanitizing test masks...")
    df_test_ready = pipeline.sanitize_masks(df_test, class_pixel_value=40)

    # Step 4: Compute dataset-specific normalization values
    custom_stats = None # use imagenet for baseline

    # Step 5: Finalize DataLoaders for the training loop
    print("Generating Dataloaders...")
    train_dl, val_dl, test_dl = pipeline.get_dataloaders(
        train_val_df=df_train_val_ready,
        test_df=df_test_ready,
        bs=16,
        img_size=512,
        custom_stats=custom_stats
    )

    # Final summary of training readiness
    print(f"\nPipeline Ready:")
    print(f" - Training batches: {len(train_dl)}")
    print(f" - Validation batches: {len(val_dl)}")
    print(f" - Testing batches: {len(test_dl)}")

    return train_dl, val_dl, test_dl, custom_stats

In [24]:
# set up baseline_model_pipeline

base_dir = os.path.normpath(os.path.join(os.getcwd(), "../"))
experiments = [
    # --- 1. Single-Domain Experiments (70/20 split -> 10% test) ---
    ({
        "train_files": ["TA_train.csv"], "val_files": ["TA_val.csv"], 
        "test_files": ["TA_test.csv"]
    }, "Single-TA"),
    
    ({
        "train_files": ["TB_train.csv"], "val_files": ["TB_val.csv"], 
        "test_files": ["TB_test.csv"]
    }, "Single-TB"),
    
    ({
        "train_files": ["TC_train.csv"], "val_files": ["TC_val.csv"], 
        "test_files": ["TC_test.csv"]
    }, "Single-TC"),

    # --- 2. Multi-Domain Experiment (Combined 70/20 -> 10% test) ---
    ({
        "train_files": ["TA_train.csv", "TB_train.csv", "TC_train.csv"],
        "val_files": ["TA_val.csv", "TB_val.csv", "TC_val.csv"],
        "test_files": ["TA_test.csv", "TB_test.csv", "TC_test.csv"]
    }, "Multi-Domain"),

    # --- 3. Domain Shift Experiments (Testing on Unseen Tunnels) ---
    # Shift to Tunnel C
    ({
        "train_files": ["TA_train.csv", "TB_train.csv"],
        "val_files": ["TA_val.csv", "TB_val.csv"],
        "test_files": ["TC_test.csv"] # TC (10%) 
    }, "Shift-TA_TB-to-TC_10pct"),
    ({
        "train_files": ["TA_train.csv", "TB_train.csv"],
        "val_files": ["TA_val.csv", "TB_val.csv"],
        "test_files": ["TC_train.csv", "TC_val.csv", "TC_test.csv"] # TC (100%) 
    }, "Shift-TA_TB-to-TC_100pct"),

    # Shift to Tunnel B
    ({
        "train_files": ["TA_train.csv", "TC_train.csv"],
        "val_files": ["TA_val.csv", "TC_val.csv"],
        "test_files": ["TB_test.csv"] # TB (10%) 
    }, "Shift-TA_TC-to-TB_10pct"),
    ({
        "train_files": ["TA_train.csv", "TC_train.csv"],
        "val_files": ["TA_val.csv", "TC_val.csv"],
        "test_files": ["TB_train.csv", "TB_val.csv", "TB_test.csv"] # TB (100%) 
    }, "Shift-TA_TC-to-TB_100pct"),

    # Shift to Tunnel A
    ({
        "train_files": ["TB_train.csv", "TC_train.csv"],
        "val_files": ["TB_val.csv", "TC_val.csv"],
        "test_files": ["TA_test.csv"] # TA (10%) 
    }, "Shift-TB_TC-to-TA_10pct"),
    ({
        "train_files": ["TB_train.csv", "TC_train.csv"],
        "val_files": ["TB_val.csv", "TC_val.csv"],
        "test_files": ["TA_train.csv", "TA_val.csv", "TA_test.csv"] # TA (100%) 
    }, "Shift-TB_TC-to-TA_100pct")
]

In [25]:
class TTDCombinedLoss(nn.Module):
    def __init__(self, ce_weight_tensor, w_ce=0.5, w_dice=0.5):
        """
        Hybrid loss combining Weighted Cross-Entropy and Dice Loss[cite: 233, 241].
        """
        super().__init__()
        self.w_ce, self.w_dice = w_ce, w_dice
        # Weighted CE handles the <0.5% crack pixel density [cite: 238, 242, 247]
        self.ce_loss = CrossEntropyLossFlat(weight=ce_weight_tensor, axis=1)
        # Dice Loss focuses on regional overlap [cite: 234, 235]
        self.dice_loss = DiceLoss(axis=1)

    def forward(self, pred, targ):
        # Cast to standard tensors to prevent fastai subclass conflicts
        pred_tensor = pred.as_subclass(torch.Tensor)
        targ_tensor = targ.as_subclass(torch.Tensor).long()
        
        ce = self.ce_loss(pred_tensor, targ_tensor)
        dice = self.dice_loss(pred_tensor, targ_tensor)
        return self.w_ce * ce + self.w_dice * dice

In [7]:
# List to store results for final analysis
all_results = []

# Use tqdm to track progress across the 10 experiments
for config, name in tqdm(experiments, desc="Running TTD Experiments"):
    
    # 1. Suppress the detailed print output from the pipeline and training loop
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        train_dl, val_dl, test_dl, custom_stats = baseline_model_pipeline(parent_dir, config)
        
        num_epoch = 100
        model = smp.Unet("resnet34", encoder_weights="imagenet", classes=2)
        model_name = f"Unet-resnet34-imagenet_{name}"
        
        optimizer = optim.AdamW(model.parameters(), lr=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=num_epoch,
            eta_min=1e-7
        )

        # Apply the weight factor of 20 to the crack class
        weights = torch.tensor([1.0, 20.0]).to(device)
        loss_fn = TTDCombinedLoss(ce_weight_tensor=weights, w_ce=0.5, w_dice=0.5)

        # Run the training loop
        history = epochs(model, model_name, device, train_dl, val_dl, loss_fn, optimizer, num_epoch, scheduler=scheduler, patience=10, save_dir="models")
        save_training_history(history, model_name, save_dir="figures")

        # 2. Reload the best model based on IoU for evaluation
        model.load_state_dict(torch.load(os.path.join("models", f"{model_name}.pth")))
        model.to(device)

        # 3. Collect detailed metrics for both Validation and Testing
        v_loss, v_iou, v_f1, v_recall, v_prec = val_loop(model, device, val_dl, loss_fn, is_test=True)
        t_loss, t_iou, t_f1, t_recall, t_prec = val_loop(model, device, test_dl, loss_fn, is_test=True)

        save_prediction_overlap(model, model_name, test_dl, device, custom_stats=custom_stats)

    # 4. Store the captured data
    all_results.append({
        "Experiment": model_name,
        "Val_Loss": v_loss, "Val_IoU": v_iou, "Val_F1": v_f1, "Val_Recall": v_recall, "Val_Prec": v_prec,
        "Test_Loss": t_loss, "Test_IoU": t_iou, "Test_F1": t_f1, "Test_Recall": t_recall, "Test_Prec": t_prec
    })

    temp_df = pd.DataFrame(all_results)
    temp_df.to_csv("TTD_baseline_results.csv", index=False)

display(pd.read_csv("TTD_baseline_results.csv"))

Running TTD Experiments:   0%|          | 0/10 [00:00<?, ?it/s]

KeyError: 'is_valid'